# Task 4: Application Development

## 1. Mount Google Drive and Check Project Files

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os


BASE_DIR = Path("/content/drive/MyDrive/CMP7005_project/reproducible")
PROJECT_DIR = BASE_DIR.parent
DATA_DIR = PROJECT_DIR / "data"
FIG_DIR = BASE_DIR / "figures"
RESULT_DIR = BASE_DIR / "results"
APP_DIR = BASE_DIR / "air_quality_app"
PAGES_DIR = APP_DIR / "pages"

FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
PAGES_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR exists:", BASE_DIR.exists())
print("DATA_DIR exists:", DATA_DIR.exists())
print("FIG_DIR exists:", FIG_DIR.exists())
print("RESULT_DIR exists:", RESULT_DIR.exists())
print("APP_DIR exists:", APP_DIR.exists())

if FIG_DIR.exists():
    print("\nSaved figures:")
    for f in sorted(FIG_DIR.iterdir()):
        if f.suffix.lower() in [".png", ".jpg", ".jpeg"]:
            print("-", f.name)


Mounted at /content/drive
BASE_DIR exists: True
DATA_DIR exists: True
FIG_DIR exists: True
RESULT_DIR exists: True
APP_DIR exists: True

Saved figures:
- 01_pollutant_distributions.png
- 02_meteorological_variable_distributions.png
- 03_pm25_station_median_iqr.png
- 04_pm25_distribution_by_station_kde.png
- 05_mean_pm25_urban_suburban.png
- 06_pm25_vs_temp_density_trend.png
- 07_no2_vs_o3_density_trend.png
- 08_correlation_heatmap.png
- 09_corr_with_pm25.png
- 10_pm25_seasonal_pattern_by_month.png
- 11_monthly_pm25_urban_suburban_improved.png
- 12_seasonal_pm25_urban_suburban_simple.png
- 13_diurnal_pm25_pattern_day_night_simplified.png
- 14_pm25_air_quality_levels_pie.png
- 15_average_pm25_weekday_weekend_pie.png
- 16_wind_direction_frequency_polar_with_regions_spacious.png
- 17_pm25_decomposition_trend_monthly_seasonal.png
- 18_eda_mean_pollutant_levels_by_station_heatmap.png
- 19_eda_pollutant_composition_by_station_combined.png
- 20_eda_yearly_average_pollutants_combined.png


## 2. Install Required GUI Package

In [2]:
!pip install streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 86.0 MB/s eta 0:00:00


## 3. Create Application Folder Structure

In [3]:
PAGES_DIR.mkdir(parents=True, exist_ok=True)
print("Created Streamlit app folder:", APP_DIR)

Created Streamlit app folder: /content/drive/MyDrive/CMP7005_project/reproducible/air_quality_app


## 4. Write Shared Configuration File

In [4]:
%%writefile /content/drive/MyDrive/CMP7005_project/reproducible/air_quality_app/config.py
from pathlib import Path
import pandas as pd
import streamlit as st

APP_DIR = Path(__file__).resolve().parent
BASE_DIR = APP_DIR.parent
PROJECT_DIR = BASE_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
FIG_DIR = BASE_DIR / "figures"
RESULT_DIR = BASE_DIR / "results"

CLEANED_DATA_CANDIDATES = [
    RESULT_DIR / "combined_cleaned_air_quality.csv",
    DATA_DIR / "combined_cleaned_air_quality.csv",
    DATA_DIR / "combined_data.csv"
]

MODEL_RESULT_CANDIDATES = [
    RESULT_DIR / "final_model_results.csv",
    DATA_DIR / "final_model_results.csv",
    RESULT_DIR / "task3_model_results.csv",
    DATA_DIR / "task3_model_results.csv",
    RESULT_DIR / "model_results.csv",
    DATA_DIR / "model_results.csv"
]

STATION_ORDER = ["Changping", "Shunyi", "Tiantan", "Wanshouxigong"]


def first_existing_path(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return None


def safe_dataframe(data):
    """
    Make a dataframe safe for Streamlit/Arrow display.
    This prevents errors caused by mixed object columns, such as strings mixed with bool values.
    """
    if data is None:
        return None

    df_safe = data.copy()

    for col in df_safe.columns:
        if pd.api.types.is_datetime64_any_dtype(df_safe[col]):
            df_safe[col] = df_safe[col].astype(str)

    object_cols = df_safe.select_dtypes(include=["object"]).columns
    for col in object_cols:
        df_safe[col] = df_safe[col].astype(str)

    return df_safe


@st.cache_data
def load_dataset():
    path = first_existing_path(CLEANED_DATA_CANDIDATES)
    if path is None:
        return None, None

    df = pd.read_csv(path)

    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

    return df, str(path)


@st.cache_data
def load_model_results():
    path = first_existing_path(MODEL_RESULT_CANDIDATES)
    if path is None:
        return None, None

    df = pd.read_csv(path)
    return df, str(path)


def figure_path(filename: str) -> Path:
    return FIG_DIR / filename


def figure_exists(filename: str) -> bool:
    return figure_path(filename).exists()


def show_figure(filename: str, caption: str = None):
    path = figure_path(filename)
    if path.exists():
        st.image(str(path), caption=caption, width="stretch")
    else:
        st.warning(f"Figure not found: {filename}")


def format_number(value, decimals=2):
    try:
        return f"{float(value):,.{decimals}f}"
    except Exception:
        return str(value)


Writing /content/drive/MyDrive/CMP7005_project/reproducible/air_quality_app/config.py


## 5. Write Main Application Page

In [5]:
%%writefile /content/drive/MyDrive/CMP7005_project/reproducible/air_quality_app/app.py
import streamlit as st
from config import load_dataset, load_model_results

st.set_page_config(
    page_title="Beijing Air Quality Analysis",
    page_icon="☁️",
    layout="wide"
)

df, data_path = load_dataset()
results_df, results_path = load_model_results()


st.sidebar.title("CMP7005 PRAC1")
st.sidebar.write("**Project:** Beijing Air Quality Analysis")
st.sidebar.write("**Target:** PM2.5")
st.sidebar.write("**App Sections:** Dataset | Visualisation | Model Outputs")

if data_path:
    st.sidebar.success("Cleaned dataset loaded")
else:
    st.sidebar.error("Cleaned dataset not found")

if results_path:
    st.sidebar.success("Model results loaded")
else:
    st.sidebar.warning("Model results CSV not found")

st.title("☁️ Beijing Air Quality Analysis Application")

st.markdown(
    """
    This application presents the complete analytical workflow for PM2.5 prediction using the Beijing air-quality dataset.

    The analysis focuses on four representative monitoring stations:
    **Changping, Shunyi (suburban)** and **Tiantan, Wanshouxigong (urban)**, allowing comparison between different spatial environments.

    **Main sections:**

    1. **Dataset Section** – reviews the cleaned dataset, selected stations, date range, missing values, data types, and summary statistics.

    2. **Visualisation Section** – presents EDA results covering pollutant distributions, meteorological variables, correlation patterns, spatial differences between urban and suburban areas, and temporal PM2.5 variations.

    3. **Model Outputs Section** – displays regression and time-series modelling results, including RMSE comparison, prediction performance, Random Forest feature importance, and SARIMAX benchmark forecasts.
    """
)

st.divider()

col1, col2, col3, col4 = st.columns(4)

if df is not None:
    with col1:
        st.metric("Rows", f"{df.shape[0]:,}")
    with col2:
        st.metric("Columns", df.shape[1])
    with col3:
        st.metric("Stations", df["station"].nunique() if "station" in df.columns else "N/A")
    with col4:
        st.metric("Missing values", int(df.isnull().sum().sum()))
else:
    st.warning("Dataset could not be loaded. Please check the data path in config.py.")

st.divider()

st.subheader("Best Model Summary")

if results_df is not None and not results_df.empty:
    if "RMSE" in results_df.columns:
        results_df = results_df.sort_values("RMSE").reset_index(drop=True)
        best_model = results_df.iloc[0]
        st.success(
            f"Best model based on RMSE: **{best_model['Model']}** "
            f"| RMSE = {best_model['RMSE']:.3f} "
            f"| MAE = {best_model['MAE']:.3f} "
            f"| R² = {best_model['R2']:.3f}"
        )
    else:
        st.info("Model results were loaded, but the RMSE column was not found.")
else:
    st.info("Model results CSV was not found. The Model Outputs page can still show saved model figures if available.")

st.divider()

st.subheader("How to Use This Application")
st.write(
    "Use the page menu on the left to move between the dataset, EDA visualisations, and model outputs. "
    "The application is structured to match the assessment workflow and support clear navigation."
)

Writing /content/drive/MyDrive/CMP7005_project/reproducible/air_quality_app/app.py


## 6. Write Dataset Section Page

In [6]:
%%writefile /content/drive/MyDrive/CMP7005_project/reproducible/air_quality_app/pages/1_Dataset_Section.py
import pandas as pd
import streamlit as st
from config import load_dataset, safe_dataframe, STATION_ORDER

st.title("📖 Dataset Section")
st.write("This section allows users to inspect the cleaned dataset used for EDA and model building.")

df, data_path = load_dataset()

if df is None:
    st.error("Cleaned dataset not found. Please check the file path in config.py.")
    st.stop()

st.caption(f"Loaded file: {data_path}")

st.sidebar.header("Dataset Filters")

filtered_df = df.copy()

if "station" in filtered_df.columns:
    available_stations = [s for s in STATION_ORDER if s in filtered_df["station"].dropna().unique()]
    selected_stations = st.sidebar.multiselect(
        "Select station(s)",
        available_stations,
        default=available_stations
    )
    if selected_stations:
        filtered_df = filtered_df[filtered_df["station"].isin(selected_stations)]

if "datetime" in filtered_df.columns and pd.api.types.is_datetime64_any_dtype(filtered_df["datetime"]):
    min_date = filtered_df["datetime"].min().date()
    max_date = filtered_df["datetime"].max().date()
    date_range = st.sidebar.date_input(
        "Select date range",
        value=(min_date, max_date),
        min_value=min_date,
        max_value=max_date
    )

    if isinstance(date_range, tuple) and len(date_range) == 2:
        start_date, end_date = date_range
        filtered_df = filtered_df[
            (filtered_df["datetime"].dt.date >= start_date) &
            (filtered_df["datetime"].dt.date <= end_date)
        ]

col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric("Rows", f"{filtered_df.shape[0]:,}")
with col2:
    st.metric("Columns", filtered_df.shape[1])
with col3:
    st.metric("Missing values", int(filtered_df.isnull().sum().sum()))
with col4:
    if "station" in filtered_df.columns:
        st.metric("Selected stations", filtered_df["station"].nunique())
    else:
        st.metric("Selected stations", "N/A")

st.divider()

st.subheader("Dataset Preview")
preview_rows = st.slider("Rows to display", 5, 100, 10, step=5)
st.dataframe(safe_dataframe(filtered_df.head(preview_rows)), width="stretch")

st.subheader("Column Information")
column_info = pd.DataFrame({
    "Column": filtered_df.columns,
    "Data Type": filtered_df.dtypes.astype(str).values,
    "Missing Count": filtered_df.isnull().sum().values,
    "Missing %": (filtered_df.isnull().mean().values * 100).round(2)
})
st.dataframe(safe_dataframe(column_info), width="stretch")


st.subheader("Statistical Summary")
summary_df = filtered_df.describe(include="all").transpose().reset_index().rename(columns={"index": "Column"})
st.dataframe(safe_dataframe(summary_df), width="stretch")

if "station_type" in filtered_df.columns and "PM2.5" in filtered_df.columns:
    st.subheader("Mean PM2.5 by Station Type")
    type_summary = (
        filtered_df.groupby("station_type")["PM2.5"]
        .mean()
        .reset_index()
        .rename(columns={"PM2.5": "Mean PM2.5"})
    )
    st.dataframe(safe_dataframe(type_summary), width="stretch")

Writing /content/drive/MyDrive/CMP7005_project/reproducible/air_quality_app/pages/1_Dataset_Section.py


## 7. Write Visualisation Section Page

In [7]:
%%writefile /content/drive/MyDrive/CMP7005_project/reproducible/air_quality_app/pages/2_Visualisation_Section.py
import streamlit as st
from config import show_figure

st.title("🧐 Visualisation Section")
st.write("This section presents the main EDA figures generated from the cleaned dataset.")

figure_catalogue = {
    "Univariate analysis": {
        "Pollutant Distributions": {
            "file": "01_pollutant_distributions.png",
            "text": "Shows the distribution of major pollutant variables using histograms."
        },
        "Meteorological Variable Distributions": {
            "file": "02_meteorological_variable_distributions.png",
            "text": "Shows the distribution of meteorological variables such as temperature, pressure, rainfall, and wind speed."
        },
        "PM2.5 by Station: Median and IQR": {
            "file": "03_pm25_station_median_iqr.png",
            "text": "Compares the typical PM2.5 level and variability across the four selected stations."
        },
        "PM2.5 Distribution by Station": {
            "file": "04_pm25_distribution_by_station_kde.png",
            "text": "Uses KDE curves to compare PM2.5 distribution patterns across stations."
        },
        "Mean PM2.5: Urban vs Suburban": {
            "file": "05_mean_pm25_urban_suburban.png",
            "text": "Compares the average PM2.5 level between urban and suburban station groups."
        },
        "PM2.5 Air-Quality Level Distribution": {
            "file": "14_pm25_air_quality_levels_pie.png",
            "text": "Shows the proportion of PM2.5 observations in different air-quality categories."
        },
        "Wind Direction Frequency": {
            "file": "16_wind_direction_frequency_polar_with_regions_spacious.png",
            "text": "Uses a polar chart to show which wind directions appear most frequently."
        }
    },
    "Bivariate analysis": {
        "PM2.5 vs Temperature": {
            "file": "06_pm25_vs_temp_density_trend.png",
            "text": "Combines a density background with a trend line to show how PM2.5 changes across temperature ranges."
        },
        "NO2 vs O3": {
            "file": "07_no2_vs_o3_density_trend.png",
            "text": "Shows the relationship between NO2 and O3, which is useful for understanding pollutant interactions."
        },
        "Weekday vs Weekend PM2.5": {
            "file": "15_average_pm25_weekday_weekend_pie.png",
            "text": "Compares PM2.5 levels between weekdays and weekends."
        }
    },
    "Multivariate analysis": {
        "Correlation Heatmap": {
            "file": "08_correlation_heatmap.png",
            "text": "Summarises relationships between pollutants and meteorological variables."
        },
        "Correlation with PM2.5": {
            "file": "09_corr_with_pm25.png",
            "text": "Ranks variables by their correlation with PM2.5."
        },
        "Mean Pollutant Levels by Station": {
            "file": "18_eda_mean_pollutant_levels_by_station_heatmap.png",
            "text": "Compares pollutant levels across stations using a heatmap."
        },
        "Pollutant Composition by Station": {
            "file": "19_eda_pollutant_composition_by_station_combined.png",
            "text": "Shows how the selected pollutants contribute proportionally within each station."
        }
    },
    "Temporal analysis": {
        "Seasonal Pattern by Month": {
            "file": "10_pm25_seasonal_pattern_by_month.png",
            "text": "Shows monthly seasonal PM2.5 patterns and highlights higher winter pollution."
        },
        "Urban vs Suburban Monthly Trend": {
            "file": "11_monthly_pm25_urban_suburban_improved.png",
            "text": "Compares PM2.5 trends between urban and suburban station groups."
        },
        "Seasonal PM2.5: Urban vs Suburban": {
            "file": "12_seasonal_pm25_urban_suburban_simple.png",
            "text": "Summarises seasonal PM2.5 differences between urban and suburban groups."
        },
        "Diurnal PM2.5 Pattern": {
            "file": "13_diurnal_pm25_pattern_day_night_simplified.png",
            "text": "Shows how PM2.5 changes across hours of the day with day-night background periods."
        },
        "PM2.5 Decomposition: Trend and Seasonality": {
            "file": "17_pm25_decomposition_trend_monthly_seasonal.png",
            "text": "Separates PM2.5 into long-term trend and monthly seasonal effects."
        },
        "Yearly Average Pollutant Trends": {
            "file": "20_eda_yearly_average_pollutants_combined.png",
            "text": "Compares yearly average trends for multiple pollutants across stations."
        }
    }
}

selected_category = st.sidebar.radio(
    "Select visualisation category",
    list(figure_catalogue.keys())
)

selected_title = st.sidebar.selectbox(
    "Select figure",
    list(figure_catalogue[selected_category].keys())
)

figure_info = figure_catalogue[selected_category][selected_title]

st.subheader(selected_title)
st.info(figure_info["text"])
show_figure(figure_info["file"], caption=selected_title)

Writing /content/drive/MyDrive/CMP7005_project/reproducible/air_quality_app/pages/2_Visualisation_Section.py


## 8. Write Model Outputs Section Page

In [8]:
%%writefile /content/drive/MyDrive/CMP7005_project/reproducible/air_quality_app/pages/3_Model_Outputs_Section.py
import streamlit as st
from config import load_model_results, show_figure, safe_dataframe

st.title("🌳 Model Outputs Section")
st.write("This section presents model performance metrics, prediction results, and model interpretation outputs.")

results_df, results_path = load_model_results()

st.subheader("Model Performance Table")

if results_df is not None and not results_df.empty:
    st.caption(f"Loaded file: {results_path}")

    if "RMSE" in results_df.columns:
        results_df = results_df.sort_values("RMSE").reset_index(drop=True)

    st.dataframe(safe_dataframe(results_df), width="stretch")

    if all(col in results_df.columns for col in ["Model", "RMSE", "MAE", "R2"]):
        best_model = results_df.iloc[0]
        st.success(
            f"Best model: **{best_model['Model']}** | "
            f"RMSE = {best_model['RMSE']:.3f} | "
            f"MAE = {best_model['MAE']:.3f} | "
            f"R² = {best_model['R2']:.3f}"
        )
else:
    st.warning(
        "Model results CSV was not found. Please save the final model results table from the Model Building notebook."
    )

st.divider()

st.subheader("Metric Explanation")
col1, col2, col3 = st.columns(3)
with col1:
    st.markdown("**MAE**")
    st.write("Average absolute prediction error. Lower is better.")
with col2:
    st.markdown("**RMSE**")
    st.write("Penalises larger prediction errors more strongly. Lower is better.")
with col3:
    st.markdown("**R²**")
    st.write("Proportion of PM2.5 variation explained by the model. Higher is better.")

st.divider()


st.subheader("Model Visualisations")

model_figures = {
    "Prediction Comparison": {
        "file": "model_prediction_comparison.png",
        "text": "Compares actual PM2.5 values with predictions from the baseline and machine-learning models."
    },
    "RMSE Model Comparison": {
        "file": "model_comparison_rmse.png",
        "text": "Compares model error using RMSE. The model with the lowest RMSE performs best."
    },
    "Random Forest Feature Importance": {
        "file": "random_forest_feature_importance.png",
        "text": "Shows which input features were most important for the tuned Random Forest model."
    },
    "SARIMAX Benchmark Forecast": {
        "file": "sarimax_weather_forecast_all_stations.png",
        "text": "Shows the SARIMAX time-series benchmark. This compares traditional time-series modelling with machine learning."
    }
}

selected_model_figure = st.selectbox(
    "Select model figure",
    list(model_figures.keys())
)

fig_info = model_figures[selected_model_figure]
st.info(fig_info["text"])
show_figure(fig_info["file"], caption=selected_model_figure)

st.divider()

st.subheader("Modelling Conclusion")
st.write(
    "The machine-learning models, especially Random Forest and Tuned Random Forest, are expected to perform better "
    "because they can use pollutant variables, meteorological variables, time features, lag features, and rolling averages. "
    "SARIMAX is retained as a traditional time-series benchmark, but PM2.5 spikes are difficult for linear time-series models to predict accurately."
)


Writing /content/drive/MyDrive/CMP7005_project/reproducible/air_quality_app/pages/3_Model_Outputs_Section.py


## 9. Optional: Save Model Results for the GUI

In [9]:
ymodel_results_path = RESULT_DIR / "final_model_results.csv"
print("Expected model results path:", model_results_path)
print("Model results file exists:", model_results_path.exists())

Expected model results path: /content/drive/MyDrive/CMP7005_project/reproducible/results/final_model_results.csv
Model results file exists: False


## 10. Run the Streamlit Application

In [10]:
!pkill -f streamlit || true
!pkill -f lt || true
!pkill -f localtunnel || true

^C
^C
^C


In [11]:
!wget -q -O - ipv4.icanhazip.com

34.32.219.194


In [12]:
!streamlit run air_quality_app/app.py & npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: air_quality_app/app.py
⠼⠴⠦⠧Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏your url is: https://dark-melons-roll.loca.lt
^C
